In [1]:
import pickle
import networkx as nx
import pennylane as qml
from joblib import Parallel, delayed
from pennylane import numpy as np
from itertools import combinations

In [3]:
def aon_hyperedge_hamiltonian(edge):
    k = len(edge)

    minus_terms = []
    plus_terms = []

    for i in edge:
        minus_terms.append(
            qml.Identity(i) - qml.PauliZ(i)
        )

        plus_terms.append(
            qml.Identity(i) + qml.PauliZ(i)
        )

    minus_product = minus_terms[0]
    plus_product = plus_terms[0]

    for op in minus_terms[1:]:
        minus_product = minus_product @ op

    for op in plus_terms[1:]:
        plus_product = plus_product @ op

    return (
        qml.Identity(edge[0])
        - (1 / (2 ** k))
        * (minus_product + plus_product)
    )

In [4]:
def aon_cost_hamiltonian(edges):
    H = 0

    for edge in edges:
        H = H + aon_hyperedge_hamiltonian(edge)

    return H

In [5]:
dev = qml.device(
    "default.qubit",
    wires=6
)

In [6]:
@qml.qnode(dev)
def basis_cost(x):
    qml.BasisState(
        np.array(x, dtype=int),
        wires=range(6)
    )

    return qml.expval(H_C)

In [8]:
x = [0, 0, 0, 1, 1, 1]
edges = [
    (0, 1, 2),
    (3, 4, 5),
    (0, 3)
]

H_C = aon_cost_hamiltonian(edges)

print(basis_cost(x))

1.0


In [9]:
x = [0, 0, 1, 1, 0, 1]

print(basis_cost(x))

3.0


In [10]:
def dicke_state(n, weight):
    state = np.zeros(2**n, dtype=complex)

    states = list(combinations(range(n), weight))

    amplitude = 1 / np.sqrt(len(states))

    for ones in states:
        bits = np.zeros(n, dtype=int)
        bits[list(ones)] = 1

        index = int("".join(map(str, bits)), 2)

        state[index] = amplitude

    return state

In [12]:
psi0 = dicke_state(6, 3)

print(np.sum(np.abs(psi0)**2))
print(np.count_nonzero(psi0))

1.0
20


In [13]:
@qml.qnode(dev)
def dicke_cost():
    qml.StatePrep(
        psi0,
        wires=range(6)
    )

    return qml.expval(H_C)

In [14]:
print(dicke_cost())

2.4


In [15]:
def swap_mixer(beta, wires):
    n = len(wires)

    for i in range(n):
        j = (i + 1) % n

        qml.IsingXX(beta, wires=[wires[i], wires[j]])
        qml.IsingYY(beta, wires=[wires[i], wires[j]])

In [16]:
@qml.qnode(dev)
def mixer_test(beta):
    qml.StatePrep(
        psi0,
        wires=range(6)
    )

    swap_mixer(beta, list(range(6)))

    return qml.probs(wires=range(6))

In [17]:
probs = mixer_test(0.4)

In [18]:
for i, p in enumerate(probs):
    if p > 1e-10:
        bitstring = format(i, "06b")

        print(
            bitstring,
            "weight =", bitstring.count("1"),
            "prob =", p
        )

000111 weight = 3 prob = 0.014011374455415542
001011 weight = 3 prob = 0.03493393537666641
001101 weight = 3 prob = 0.04461528373485143
001110 weight = 3 prob = 0.02352518651036344
010011 weight = 3 prob = 0.046063827082311146
010101 weight = 3 prob = 0.12900895927981895
010110 weight = 3 prob = 0.058329653302108084
011001 weight = 3 prob = 0.051863860477368656
011010 weight = 3 prob = 0.06426077545655835
011100 weight = 3 prob = 0.033387144324537855
100011 weight = 3 prob = 0.033387144324537855
100101 weight = 3 prob = 0.06426077545655835
100110 weight = 3 prob = 0.051863860477368656
101001 weight = 3 prob = 0.058329653302108084
101010 weight = 3 prob = 0.12900895927981895
101100 weight = 3 prob = 0.046063827082311146
110001 weight = 3 prob = 0.02352518651036344
110010 weight = 3 prob = 0.04461528373485143
110100 weight = 3 prob = 0.03493393537666641
111000 weight = 3 prob = 0.014011374455415542
